# Add seurat annotations to anndata
Add to scanvi version so we have both seurat and scanvi

Do we want to add scanvi cluster majority for smoothing?

Removed batch suffix here

Need to remove unassigned annotaiton

In [1]:
from pathlib import Path
import os
import scanpy as sc
import pandas as pd
import numpy as np
import anndata as ad
import warnings
import session_info
import sys

In [2]:
sys.path.append(str(Path.cwd().resolve().parents[1]))

from config.paths import BASE_DIR, METADATA_DIR

# Data directories (derived from BASE_DIR)
input_dir = BASE_DIR / "data/h5ad/export_03/03a_scanvi"
adata_path = input_dir / "adata-scanvi-labels.h5ad"
seurat_labels_path = BASE_DIR / "data/rds/seurat/20251204_label_transfer_xenium.csv"

# Metadata inside Git repo (derived from REPO_DIR)
manual_annotations_path = METADATA_DIR / "manual-annotations.csv"

output_dir = BASE_DIR / "data/h5ad/export_03/03b_seurat"

In [3]:
adata = sc.read_h5ad(adata_path)
seurat_df = pd.read_csv(seurat_labels_path)
manual_df = pd.read_csv(manual_annotations_path)

# Add label transfer annotations

In [4]:
# 1 — Make a copy and set cell barcodes as index
labels = seurat_df.copy()
labels.index = labels["Unnamed: 0"]
labels = labels.drop(columns=["Unnamed: 0"])

# 2 — Keep only rows that actually exist in adata
shared = labels.index.intersection(adata.obs_names)
labels = labels.loc[shared]

# 3 — Reindex so order matches adata.obs exactly
labels = labels.reindex(adata.obs_names)

# 4 — Add all columns into adata.obs
adata.obs = adata.obs.join(labels)

In [5]:
adata.obs

,cell_id,x_centroid,y_centroid,transcript_counts,control_probe_counts,genomic_control_counts,control_codeword_counts,unassigned_codeword_counts,deprecated_codeword_counts,total_counts,...,prediction.score.Vascular_endothelial_1,prediction.score.Vascular_endothelial_2,prediction.score.Macrophage_1,prediction.score.Cycling.SGC,prediction.score.Pericyte,prediction.score.Macrophage_2,prediction.score.Fibroblast_2,prediction.score.Angiogenic.EC,prediction.score.Fibroblast_1,prediction.score.max
aaagciof-1-0,aaagciof-1,634.049133,299.172485,347,0,0,0,0,0,347.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.386594
aaailodo-1-0,aaailodo-1,609.569763,308.822144,2415,0,0,0,0,0,2415.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.876126
aaanacfp-1-0,aaanacfp-1,634.638367,309.968719,1937,0,0,0,0,0,1937.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.997552
aaaohjeo-1-0,aaaohjeo-1,627.379822,297.124298,479,0,0,0,0,0,479.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.554047
aabfmjal-1-0,aabfmjal-1,621.577148,367.691040,402,0,0,0,0,0,402.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.733216
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
oiijogbo-1-7,oiijogbo-1,4271.075684,1051.913696,56,0,0,0,0,0,56.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.750928
oijefoep-1-7,oijefoep-1,4231.781738,913.188904,45,0,0,0,0,0,45.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
oilegbah-1-7,oilegbah-1,3286.040527,3637.353027,66,0,0,0,0,0,66.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.000000
oilfilnk-1-7,oilfilnk-1,3342.463623,3643.193115,50,0,0,0,0,0,50.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
barcodes = [
    "apeeemip-1-1",
    "bpoikicj-1-1",
    "cbhmmeof-1-1",
    "akcoadof-1-2",
    "bgmdeeeb-1-2",
    "hjjgalia-1-2",
    "hjjkhdhk-1-2",
    "mnhcfodp-1-2",
    "bmjgagok-1-3",
    "ijnggkoc-1-3",
    "llknggjo-1-3",
    "llmfnafp-1-3",
    "llmgffik-1-3"
]

In [7]:
cols_to_view = ["scanvi_labels", "predicted.id"]

obs_subset = adata.obs.loc[
    adata.obs.index.intersection(barcodes),
    cols_to_view
]

obs_subset

,scanvi_labels,predicted.id
apeeemip-1-1,Vascular_endothelial_1,Vascular_endothelial_1
bpoikicj-1-1,CGRP-Gamma,CGRP-Gamma
cbhmmeof-1-1,CGRP-Gamma,CGRP-Gamma
akcoadof-1-2,CGRP-Gamma,CGRP-Gamma
bgmdeeeb-1-2,CGRP-Gamma,CGRP-Gamma
hjjgalia-1-2,CGRP-Gamma,CGRP-Gamma
hjjkhdhk-1-2,CGRP-Gamma,CGRP-Eta
mnhcfodp-1-2,CGRP-Gamma,CGRP-Gamma
bmjgagok-1-3,CGRP-Gamma,CGRP-Gamma
ijnggkoc-1-3,CGRP-Gamma,CGRP-Gamma


# Add annotations for qualitative confidence mappings

In [8]:
manual_df

,cell_id,sample_id,confidence
0,aallcopa-1,TMA00303,Middle
1,acpnmbio-1,TMA00303,Middle
2,cbhmmeof-1,TMA00304,High
3,bpoikicj-1,TMA00304,High
4,apeeemip-1,TMA00304,Middle
5,agbmnepe-1,TMA00304,High
6,hjjgalia-1,TMA00305,High
7,hjjkhdhk-1,TMA00305,High
8,mnhcfodp-1,TMA00305,High
9,akcoadof-1,TMA00305,High


In [9]:
adata.obs = adata.obs.merge(
    manual_df[['cell_id', 'sample_id', 'confidence']],
    on=['cell_id', 'sample_id'],
    how='left'
)

adata.obs

,cell_id,x_centroid,y_centroid,transcript_counts,control_probe_counts,genomic_control_counts,control_codeword_counts,unassigned_codeword_counts,deprecated_codeword_counts,total_counts,...,prediction.score.Vascular_endothelial_2,prediction.score.Macrophage_1,prediction.score.Cycling.SGC,prediction.score.Pericyte,prediction.score.Macrophage_2,prediction.score.Fibroblast_2,prediction.score.Angiogenic.EC,prediction.score.Fibroblast_1,prediction.score.max,confidence
0,aaagciof-1,634.049133,299.172485,347,0,0,0,0,0,347.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.386594,NaN
1,aaailodo-1,609.569763,308.822144,2415,0,0,0,0,0,2415.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.876126,NaN
2,aaanacfp-1,634.638367,309.968719,1937,0,0,0,0,0,1937.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.997552,NaN
3,aaaohjeo-1,627.379822,297.124298,479,0,0,0,0,0,479.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.554047,NaN
4,aabfmjal-1,621.577148,367.691040,402,0,0,0,0,0,402.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.733216,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
146271,oiijogbo-1,4271.075684,1051.913696,56,0,0,0,0,0,56.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.750928,NaN
146272,oijefoep-1,4231.781738,913.188904,45,0,0,0,0,0,45.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
146273,oilegbah-1,3286.040527,3637.353027,66,0,0,0,0,0,66.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.000000,NaN
146274,oilfilnk-1,3342.463623,3643.193115,50,0,0,0,0,0,50.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
adata.obs.index = adata.obs["cell_id"].astype(str)
adata.obs.index.name = None
adata.obs

,cell_id,x_centroid,y_centroid,transcript_counts,control_probe_counts,genomic_control_counts,control_codeword_counts,unassigned_codeword_counts,deprecated_codeword_counts,total_counts,...,prediction.score.Vascular_endothelial_2,prediction.score.Macrophage_1,prediction.score.Cycling.SGC,prediction.score.Pericyte,prediction.score.Macrophage_2,prediction.score.Fibroblast_2,prediction.score.Angiogenic.EC,prediction.score.Fibroblast_1,prediction.score.max,confidence
aaagciof-1,aaagciof-1,634.049133,299.172485,347,0,0,0,0,0,347.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.386594,NaN
aaailodo-1,aaailodo-1,609.569763,308.822144,2415,0,0,0,0,0,2415.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.876126,NaN
aaanacfp-1,aaanacfp-1,634.638367,309.968719,1937,0,0,0,0,0,1937.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.997552,NaN
aaaohjeo-1,aaaohjeo-1,627.379822,297.124298,479,0,0,0,0,0,479.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.554047,NaN
aabfmjal-1,aabfmjal-1,621.577148,367.691040,402,0,0,0,0,0,402.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.733216,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
oiijogbo-1,oiijogbo-1,4271.075684,1051.913696,56,0,0,0,0,0,56.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.750928,NaN
oijefoep-1,oijefoep-1,4231.781738,913.188904,45,0,0,0,0,0,45.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
oilegbah-1,oilegbah-1,3286.040527,3637.353027,66,0,0,0,0,0,66.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.000000,NaN
oilfilnk-1,oilfilnk-1,3342.463623,3643.193115,50,0,0,0,0,0,50.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
adata.obs.confidence.value_counts()

confidence
High      17
Middle     4
Low        1
Name: count, dtype: int64

In [12]:
adata.obs = adata.obs.rename(columns={"predicted.id": "seurat_labels"})

In [13]:
adata.obs.loc[adata.obs["confidence"].notna(), ["cell_id", "sample_id", "confidence", "scanvi_labels", "seurat_labels"]]

,cell_id,sample_id,confidence,scanvi_labels,seurat_labels
aallcopa-1,aallcopa-1,TMA00303,Middle,CGRP-Gamma,CGRP-Gamma
acpnmbio-1,acpnmbio-1,TMA00303,Middle,CGRP-Gamma,CGRP-Gamma
agbmnepe-1,agbmnepe-1,TMA00304,High,CGRP-Gamma,CGRP-Gamma
apeeemip-1,apeeemip-1,TMA00304,Middle,Vascular_endothelial_1,Vascular_endothelial_1
bpoikicj-1,bpoikicj-1,TMA00304,High,CGRP-Gamma,CGRP-Gamma
cbhmmeof-1,cbhmmeof-1,TMA00304,High,CGRP-Gamma,CGRP-Gamma
akcoadof-1,akcoadof-1,TMA00305,High,CGRP-Gamma,CGRP-Gamma
bgmdeeeb-1,bgmdeeeb-1,TMA00305,High,CGRP-Gamma,CGRP-Gamma
hjjgalia-1,hjjgalia-1,TMA00305,High,CGRP-Gamma,CGRP-Gamma
hjjkhdhk-1,hjjkhdhk-1,TMA00305,High,CGRP-Gamma,CGRP-Eta


# Export

In [14]:
filename = os.path.join(output_dir, 'adata-seurat-labels.h5ad')
os.makedirs(os.path.dirname(filename), exist_ok = True)

adata.write_h5ad(filename, compression='gzip')